# Notebook 01 — Exploratory Data Analysis

**Project:** Predict Energy Consumption — Seattle Building Benchmarking  
**Input:** `data/raw/2016_Building_Energy_Benchmarking.csv`  
**Leads to:** `02_data_cleaning_and_features.ipynb`

The city of Seattle aims to reach **carbon neutrality by 2050**. Measuring energy consumption
in every non-residential building through field surveys is expensive and slow. This project
builds a predictive model that estimates **site energy use (kBtu)** and **GHG emissions**
from structural building attributes alone — so planners can prioritise retrofits city-wide.

This notebook explores the **raw** dataset without making any changes to it.
The goal is to answer: *What are we working with?*

---

**Contents**
1. Dataset overview
2. Statistical overview
3. Key visual patterns — energy by building characteristics


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)


## 1. Dataset Overview

The Seattle 2016 Building Energy Benchmarking dataset was collected as part of the city's
mandatory energy reporting programme. It covers **non-residential and multifamily buildings**
above a certain size threshold.

We load it here to examine its shape, column names, and raw values before any transformation.


In [2]:
import pandas as pd

# Load the dataset
data = pd.read_csv(r'../data/raw/2016_Building_Energy_Benchmarking.csv')

# Explore the data
print(data.head())
print(data.info())

   OSEBuildingID  DataYear    BuildingType PrimaryPropertyType  \
0              1      2016  NonResidential               Hotel   
1              2      2016  NonResidential               Hotel   
2              3      2016  NonResidential               Hotel   
3              5      2016  NonResidential               Hotel   
4              8      2016  NonResidential               Hotel   

                  PropertyName          Address     City State  ZipCode  \
0         Mayflower park hotel    405 Olive way  Seattle    WA  98101.0   
1              Paramount Hotel  724 Pine street  Seattle    WA  98101.0   
2      5673-The Westin Seattle  1900 5th Avenue  Seattle    WA  98101.0   
3                    HOTEL MAX   620 STEWART ST  Seattle    WA  98101.0   
4  WARWICK SEATTLE HOTEL (ID8)    401 LENORA ST  Seattle    WA  98121.0   

  TaxParcelIdentificationNumber  ...  Electricity(kWh) Electricity(kBtu)  \
0                    0659000030  ...      1.156514e+06         3946027.0   


## 2. Statistical Overview

A `describe()` on the numeric columns reveals the **scale**, **spread**, and
**extreme values** of each variable. Key things to note:
- Energy columns span several orders of magnitude — the distribution is heavily right-skewed
- `ENERGYSTARScore` has significant missingness (~35% of buildings lack it)
- `YearBuilt` ranges from 1900 to 2015 — building age will be a useful feature


In [3]:
data.describe()

,OSEBuildingID,DataYear,ZipCode,CouncilDistrictCode,Latitude,Longitude,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,...,SiteEnergyUse(kBtu),SiteEnergyUseWN(kBtu),SteamUse(kBtu),Electricity(kWh),Electricity(kBtu),NaturalGas(therms),NaturalGas(kBtu),Comments,TotalGHGEmissions,GHGEmissionsIntensity
count,3376.000000,3376.0,3360.000000,3376.000000,3376.000000,3376.000000,3376.000000,3368.000000,3376.000000,3.376000e+03,...,3.371000e+03,3.370000e+03,3.367000e+03,3.367000e+03,3.367000e+03,3.367000e+03,3.367000e+03,0.0,3367.000000,3367.000000
mean,21208.991114,2016.0,98116.949107,4.439277,47.624033,-122.334795,1968.573164,1.106888,4.709123,9.483354e+04,...,5.403667e+06,5.276726e+06,2.745959e+05,1.086639e+06,3.707612e+06,1.368505e+04,1.368505e+06,NaN,119.723971,1.175916
std,12223.757015,0.0,18.615205,2.120625,0.047758,0.027203,33.088156,2.108402,5.494465,2.188376e+05,...,2.161063e+07,1.593879e+07,3.912173e+06,4.352478e+06,1.485066e+07,6.709781e+04,6.709781e+06,NaN,538.832227,1.821452
min,1.000000,2016.0,98006.000000,1.000000,47.499170,-122.414250,1900.000000,0.000000,0.000000,1.128500e+04,...,0.000000e+00,0.000000e+00,0.000000e+00,-3.382680e+04,-1.154170e+05,0.000000e+00,0.000000e+00,NaN,-0.800000,-0.020000
25%,19990.750000,2016.0,98105.000000,3.000000,47.599860,-122.350662,1948.000000,1.000000,2.000000,2.848700e+04,...,9.251286e+05,9.701822e+05,0.000000e+00,1.874229e+05,6.394870e+05,0.000000e+00,0.000000e+00,NaN,9.495000,0.210000
50%,23112.000000,2016.0,98115.000000,4.000000,47.618675,-122.332495,1975.000000,1.000000,4.000000,4.417500e+04,...,1.803753e+06,1.904452e+06,0.000000e+00,3.451299e+05,1.177583e+06,3.237538e+03,3.237540e+05,NaN,33.920000,0.610000
75%,25994.250000,2016.0,98122.000000,7.000000,47.657115,-122.319407,1997.000000,1.000000,5.000000,9.099200e+04,...,4.222455e+06,4.381429e+06,0.000000e+00,8.293178e+05,2.829632e+06,1.189033e+04,1.189034e+06,NaN,93.940000,1.370000
max,50226.000000,2016.0,98272.000000,7.000000,47.733870,-122.220966,2015.000000,111.000000,99.000000,9.320156e+06,...,8.739237e+08,4.716139e+08,1.349435e+08,1.925775e+08,6.570744e+08,2.979090e+06,2.979090e+08,NaN,16870.980000,34.090000


## 3. Key Patterns — Energy by Building Characteristics

Before any cleaning we visualise the most informative relationships:
- **Energy use by building type** — are some types systematically higher consumers?
- **Energy vs ENERGY STAR Score** — does certification correlate with consumption?
- **Energy by largest use type** — hotel vs office vs school etc.
- **Energy vs year built** — do older buildings use more energy?

> Log scale is used on the y-axis. Fliers are hidden to focus on the central distribution.


## Key Observations

| Finding | Implication |
|---|---|
| Energy values are right-skewed across all building types | Log transform may help models |
| ENERGY STAR Score correlates inversely with energy use | Useful feature — but missing for ~35% of buildings |
| Hotels and campuses are high consumers; schools are lower | Building use type is a strong signal |
| Older buildings tend to use slightly more energy | Year built / Age is worth engineering |

**Next:** `02_data_cleaning_and_features.ipynb` — clean the data, engineer features, and
produce the modelling-ready dataset.
